In [ ]:
from pathlib import Path

UCF_ROOT = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/"
HMDB_ROOT = "/kaggle/input/datasets/kodurujagruthaaditya/hmdb51/hmdb51_org/"

# ALT_PKG: update this to wherever your alt package folder lives
ALT_PKG = "/kaggle/input/datasets/darsankrishna12/alt-optionb-code/alt "  # ← set this to your actual alt package path

print("UCF_ROOT :", UCF_ROOT)
print("HMDB_ROOT:", HMDB_ROOT)
print("ALT_PKG  :", ALT_PKG)

In [ ]:
import shutil, os

TARGET = "/kaggle/working/alt"

# Remove previous copied project only
if os.path.exists(TARGET):
    shutil.rmtree(TARGET)

# Copy dataset folder
shutil.copytree(ALT_PKG, TARGET)

# Enter project
os.chdir(TARGET)

print("Working dir:", os.getcwd())
print("Contents:", os.listdir(".")[:10])

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "git+https://github.com/openai/CLIP.git",
    "git+https://github.com/facebookresearch/ToMe.git",
    "decord",
], check=True)

import sys, os
from pathlib import Path

ALT_DST = "/kaggle/working/alt"
sys.path.insert(0, ALT_DST)
os.chdir(ALT_DST)
print("CWD:", os.getcwd())

# Quick import + upgraded-code verification
from models.alt import ALT
from training.trainer import Trainer
clip_encoder_path = Path("/kaggle/working/alt/models/clip_encoder.py")
clip_encoder_text = clip_encoder_path.read_text(encoding="utf-8")
print("Imports OK")
print("clip_encoder.py mtime:", clip_encoder_path.stat().st_mtime)
print("Has _merge_tokens:", "def _merge_tokens" in clip_encoder_text)


In [ ]:
import os
import yaml
from pprint import pprint

CFG_PATH = "/kaggle/working/alt/configs/alt_b16.yaml"
cfg = yaml.safe_load(open(CFG_PATH, "r"))

cfg["clip_model"] = "ViT-B/16"
cfg["corpus_path"] = "/kaggle/working/alt/corpus/corpus_embeddings.pt"
cfg["label_file"] = "/kaggle/working/alt/corpus/all_labels_ordered.json"
cfg["train_json"] = "/kaggle/working/alt/corpus/train_labels.json"
cfg["val_json"] = "/kaggle/working/alt/corpus/val_labels.json"
cfg["data_roots"] = {"ucf": UCF_ROOT, "hmdb": HMDB_ROOT}

cfg["corpus_build_if_missing"] = True
cfg["corpus_rebuild"] = True
cfg["corpus_use_t5_filter"] = True
cfg["corpus_use_lesk"] = False
cfg["corpus_t5_model"] = "t5-small"

# Optional runtime overrides (keep defaults unless experimenting)
# cfg["tome_r"] = 4
# cfg["tome_layers"] = None
# cfg["tau"] = 1.0
# cfg["alignment_topk"] = None
# cfg["early_stopping_patience"] = 3

with open(CFG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config patched:")
pprint(cfg)

print("\nPath checks:")
for k in ["corpus_path", "label_file", "train_json", "val_json"]:
    print(f"  {k}: {cfg[k]} -> exists={os.path.exists(cfg[k])}")


In [ ]:
import yaml
from data.video_dataset import VideoDataset
from data.transforms import get_train_transforms, get_val_transforms

cfg = yaml.safe_load(open("/kaggle/working/alt/configs/alt_b16.yaml", "r"))

train_ds = VideoDataset(
    cfg["train_json"],
    cfg["data_roots"],
    cfg["num_frames"],
    get_train_transforms(cfg["frame_size"]),
    mode="train",
)
val_ds = VideoDataset(
    cfg["val_json"],
    cfg["data_roots"],
    cfg["num_frames"],
    get_val_transforms(cfg["frame_size"]),
    mode="val",
)

# Verification: dataset creation succeeds without monkey-patch
print("VideoDataset uses built-in resolver:", hasattr(train_ds, "_resolve_path"))
print("train_ds size:", len(train_ds), "val_ds size:", len(val_ds))
print("Resolved sample path:", train_ds._resolve_path(train_ds.entries[0]))


In [ ]:
import json
import subprocess
from collections import defaultdict
from pathlib import Path

# Load HMDB val split (not the combined val_json)
hmdb_val_path = "/kaggle/working/alt/corpus/hmdb_val_split1.json"
val_entries   = json.load(open(hmdb_val_path, "r"))

# Group by label, take first 2 classes, up to 5 videos each
bucket = defaultdict(list)
for e in val_entries:
    bucket[e["label"]].append(e)

selected = []
for label in list(bucket.keys())[:2]:
    selected.extend(bucket[label][:5])

tiny_split = "/kaggle/working/alt/corpus/hmdb_val_tiny.json"
with open(tiny_split, "w") as f:
    json.dump(selected, f)

# Build a dummy checkpoint for the sanity run
import torch
from models.alt import ALT
S = torch.load(cfg["corpus_path"], map_location="cpu").float()
tmp_model = ALT(cfg, S)   # uses cfg, includes ToMe, alignment, adapter
tmp_ckpt = "/kaggle/working/runs/alt_b16/init_sanity.pt"
Path("/kaggle/working/runs/alt_b16").mkdir(parents=True, exist_ok=True)
torch.save({"epoch": 0, "best_acc": 0.0, "model_state": tmp_model.state_dict()}, tmp_ckpt)

print("Sanity split size:", len(selected))
if len(selected) == 0:
    raise RuntimeError("No HMDB samples found for sanity split.")

# Run eval on the tiny split
subprocess.run([
    "python", "/kaggle/working/alt/eval.py",
    "--config",       "/kaggle/working/alt/configs/alt_b16.yaml",
    "--checkpoint",   tmp_ckpt,
    "--split_json",   tiny_split,
    "--label_offset", "101",
    "--n_classes",    "51",
], check=True)
print("Tiny zero-shot sanity check completed.")

In [ ]:
import os
required = [
    '/kaggle/working/alt/corpus/corpus_embeddings.pt',
    '/kaggle/working/alt/corpus/all_labels_ordered.json',
    '/kaggle/working/alt/corpus/train_labels.json',
    '/kaggle/working/alt/corpus/val_labels.json',
    '/kaggle/working/alt/corpus/hmdb_val_split1.json',
    '/kaggle/working/alt/models/alt.py',
    '/kaggle/working/alt/models/clip_encoder.py',
    '/kaggle/working/alt/models/alignment.py',
    '/kaggle/working/alt/models/video_adapter.py',
    '/kaggle/working/alt/data/video_dataset.py',
    '/kaggle/working/alt/data/transforms.py',
    '/kaggle/working/alt/training/trainer.py',
    '/kaggle/working/alt/training/loss.py',
    '/kaggle/working/alt/train.py',
    '/kaggle/working/alt/eval.py',
    '/kaggle/working/alt/configs/alt_b16.yaml',
]
for f in required:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")

In [ ]:
import yaml, torch
from models.alt import ALT

cfg = yaml.safe_load(open('/kaggle/working/alt/configs/alt_b16.yaml', 'r'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

S = torch.load(cfg['corpus_path'], map_location='cpu').float()
model = ALT(cfg, S).to(device).eval()

with torch.no_grad():
    dummy = torch.randn(2, cfg['num_frames'], 3, cfg['frame_size'], cfg['frame_size'], device=device)
    token_feats = model.encoder(dummy)

print('Encoder token shape:', tuple(token_feats.shape))
token_count = token_feats.shape[2]
if cfg.get('tome_r', 0) > 0 and token_count >= 197:
    raise RuntimeError(f'ToMe seems inactive: got {token_count} tokens (expected < 197).')
print('ToMe verification passed.')


In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working/alt")
os.chdir("/kaggle/working/alt")

import yaml, torch
from models.alt import ALT
from data.video_dataset import VideoDataset
from data.transforms import get_train_transforms, get_val_transforms
from training.trainer import Trainer
from training.loss import build_label_embeddings

cfg = yaml.safe_load(open("configs/alt_b16.yaml", "r"))
# Recommended smoke run before full training:
# cfg["epochs"] = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

S = torch.load(cfg["corpus_path"], map_location="cpu").float()
model = ALT(cfg, S)
train_ds = VideoDataset(cfg["train_json"], cfg["data_roots"], cfg["num_frames"], get_train_transforms(cfg["frame_size"]), "train")
val_ds = VideoDataset(cfg["val_json"], cfg["data_roots"], cfg["num_frames"], get_val_transforms(cfg["frame_size"]), "val")
C = build_label_embeddings(cfg["label_file"], device)
torch.cuda.empty_cache()
print("Cleared CUDA cache after label embeddings")

trainer = Trainer(model, cfg, train_ds, val_ds, C, device)
trainer.run(start_epoch=0, output_dir="/kaggle/working/runs/alt_b16", log_dir="/kaggle/working/runs/alt_b16")


In [ ]:
import glob
import torch

candidates = sorted(set(
    glob.glob("/kaggle/working/runs/alt_b16/latest.pt") +
    glob.glob("/kaggle/input/*/latest.pt") +
    glob.glob("/kaggle/input/**/*latest.pt", recursive=True)
))
print("Found checkpoints:")
for c in candidates:
    print(" -", c)

if candidates:
    ranked = []
    for path in candidates:
        try:
            ck = torch.load(path, map_location="cpu")
            ranked.append((int(ck.get("epoch", -1)), path))
        except Exception as e:
            print(f"Skipping unreadable checkpoint {path}: {e}")
    ranked.sort(key=lambda x: x[0])
    best_epoch, best_path = ranked[-1]
    print(f"Resuming from: {best_path} (epoch={best_epoch})")

    trainer = Trainer(model, cfg, train_ds, val_ds, C, device)
    last_ep = trainer.load_checkpoint(best_path)
    print(f"Loaded checkpoint epoch={last_ep}")
    trainer.run(start_epoch=last_ep + 1, output_dir="/kaggle/working/runs/alt_b16", log_dir="/kaggle/working/runs/alt_b16")
else:
    print("No checkpoint found to resume.")


In [ ]:
import os
import shutil
import subprocess
import torch

subprocess.run([
    "python", "/kaggle/working/alt/eval.py",
    "--config", "/kaggle/working/alt/configs/alt_b16.yaml",
    "--checkpoint", "/kaggle/working/runs/alt_b16/best.pt",
    "--split_json", "/kaggle/working/alt/corpus/hmdb_val_split1.json",
    "--label_offset", "101",
    "--n_classes", "51",
], check=True)

torch.cuda.empty_cache()
print("Cleared CUDA cache after evaluation")

artifacts = [
    ("/kaggle/working/runs/alt_b16/best.pt", "/kaggle/working/best.pt"),
    ("/kaggle/working/runs/alt_b16/metrics.csv", "/kaggle/working/metrics.csv"),
    ("/kaggle/working/alt/configs/alt_b16.yaml", "/kaggle/working/alt_b16.yaml"),
]
for src, dst in artifacts:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Saved: {dst}")
    else:
        print(f"Missing (skip): {src}")
